In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from cleaning import clean_data, NUM_COLS, CAT_COLS, REF_YEAR

RS = 42  # fixed random state used everywhere

raw = pd.read_csv("data/used_cars.csv")
df = clean_data(raw)
df.to_csv("data/used_cars_clean.csv", index=False)

print("raw:", raw.shape)
print("clean:", df.shape)

raw: (4009, 12)
clean: (4001, 11)


In [2]:
X = df.drop(columns="price")
y = df["price"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RS)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RS)

print("train:", X_train.shape[0])
print("val:  ", X_val.shape[0])
print("test: ", X_test.shape[0])

train: 2800
val:   600
test:  601


In [3]:
# how many missing values before filling (for the report)
print(X_train[NUM_COLS].isna().sum())

# learn the fill values from train only
medians = X_train[NUM_COLS].median()
modes = X_train[CAT_COLS].mode().iloc[0]

# apply the same values to all three sets
X_train = X_train.fillna(medians).fillna(modes)
X_val = X_val.fillna(medians).fillna(modes)
X_test = X_test.fillna(medians).fillna(modes)

print("missing left in train:", X_train.isna().sum().sum())
print(medians)

milage             0
car_age            0
horsepower       542
engine_liters     31
cylinders        172
dtype: int64
missing left in train: 0
milage           53000.0
car_age              8.0
horsepower         310.0
engine_liters        3.5
cylinders            6.0
dtype: float64


In [4]:
scaler = StandardScaler()
scaler.fit(X_train[NUM_COLS])

encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False)
encoder.fit(X_train[CAT_COLS])

print("categories learned per column:")
for col, cats in zip(CAT_COLS, encoder.categories_):
    print(col, len(cats))

categories learned per column:
brand 54
fuel_type 7
transmission_type 5
accident 3
clean_title 2


In [5]:
def prepare(data):
    num = scaler.transform(data[NUM_COLS])
    cat = encoder.transform(data[CAT_COLS])
    return np.hstack([num, cat])

X_train_p = prepare(X_train)
X_val_p = prepare(X_val)
X_test_p = prepare(X_test)

print("columns before encoding:", X_train.shape[1])
print("features after encoding:", X_train_p.shape[1])
print("train:", X_train_p.shape, "val:", X_val_p.shape, "test:", X_test_p.shape)
print("NaN left:", np.isnan(X_train_p).sum())

columns before encoding: 10
features after encoding: 62
train: (2800, 62) val: (600, 62) test: (601, 62)
NaN left: 0


In [6]:
os.makedirs("../models", exist_ok=True)

# everything the GUI needs to prepare a new car in the same way
prep = {
    "medians": medians,
    "modes": modes,
    "scaler": scaler,
    "encoder": encoder,
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}
joblib.dump(prep, "../models/part_a_scaler.joblib")

# dropdown choices for the GUI
options = {col: sorted(df[col].dropna().unique().tolist()) for col in CAT_COLS}
options["ref_year"] = REF_YEAR
joblib.dump(options, "../models/part_a_options.joblib")

# the ready splits for the models notebook
joblib.dump((X_train_p, X_val_p, X_test_p, y_train, y_val, y_test), "data/processed_splits.joblib")

print("saved")
print(options["fuel_type"])
print(options["transmission_type"])

saved
['Diesel', 'E85 Flex Fuel', 'Electric', 'Gasoline', 'Hybrid', 'Plug-In Hybrid', 'Unknown']
['Automatic', 'CVT', 'Dual-Shift', 'Manual', 'Other']
